In [47]:
from selenium import webdriver as wd
from dotenv import load_dotenv
import os
from selenium.webdriver.common.by import By
import pandas as pd

In [48]:
# 번외 : 확장성 (다른 지역) 을 고려한 함수
def get_target_sido_list(default_val=['서울']):
    """.env 파일에서 TARGET_SIDO_LIST를 읽어와 리스트로 반환하는 함수
    """
    raw_target_sido = os.getenv('TARGET_SIDO_LIST')
    
    if not raw_target_sido or not raw_target_sido.strip(): # 공백 처리 추가
        print(f"[CONFIG] '.env'에 설정된 지역이 없습니다. 기본 지역을 사용합니다: {default_val}")
        return default_val
    target_list = [name.strip() for name in raw_target_sido.split(',')]
    print(f"[CONFIG] 수집 대상 지역 확정: {target_list} (출처: .env)")
    return target_list

In [49]:
# dotenv로 환경변수 불러오기
load_dotenv()
STARBUCKS_URL        = os.getenv('STARBUCKS_URL')
WAIT_TIME            = os.getenv('WAIT_TIME')
REGION_SEARCH_BUTTON = os.getenv('REGION_SEARCH_BUTTON')
SLEEP_TIME           = os.getenv('SLEEP_TIME')
TARGET_SIDO_LIST     = get_target_sido_list()
#print(TARGET_SIDO_LIST)

[CONFIG] 수집 대상 지역 확정: ['서울'] (출처: .env)


In [50]:
print(f"[SYSTEM] 자동화 브라우저(Chrome)를 기동합니다.")
driver = wd.Chrome()

print(f"[CONNECT] 스타벅스 매장 찾기 페이지로 이동합니다.")
driver.get(STARBUCKS_URL)
# 안정성을 위해 화면 로드될 때까지 대기
print(f"[CONFIG] 페이지 요소 로딩 대기 시간 설정: {WAIT_TIME}초")
driver.implicitly_wait(WAIT_TIME)

[SYSTEM] 자동화 브라우저(Chrome)를 기동합니다.
[CONNECT] 스타벅스 매장 찾기 페이지로 이동합니다.
[CONFIG] 페이지 요소 로딩 대기 시간 설정: 5초


In [51]:
import time

def force_click_region_tab(driver):
    """커피콩 로딩 오버레이를 무시하고 '지역 검색' 탭을 강제 클릭하는 함수
    """
    try:
        # 화면에서 지역검색 클릭
        region_tab = driver.find_element(By.LINK_TEXT, "지역 검색")
        
        print(f"[BYPASS] 로딩 오버레이 무시: '지역 검색' 탭 강제 클릭 실행")
        
        # 3. JS로 강제 클릭
        driver.execute_script("arguments[0].click();", region_tab)
        
    except Exception as e:
        print(f"[ERROR] 지역 검색 탭 클릭 실패: {e}")

In [52]:
force_click_region_tab(driver)

[BYPASS] 로딩 오버레이 무시: '지역 검색' 탭 강제 클릭 실행


In [53]:
# CSS_SELECTOR 
sidos = driver.find_elements(By.CSS_SELECTOR, "a.set_sido_cd_btn")
sidos_value = [
    {
        'code' : sido.get_attribute("data-sidocd").strip(),
        'name' : sido.text
    } 
    for sido in sidos
]
print(f"[SYSTEM] 전국 시/도 정보 수집 완료: 총 {len(sidos_value)}개 지역 매핑 성공")
#len(sidos)

[SYSTEM] 전국 시/도 정보 수집 완료: 총 17개 지역 매핑 성공


In [54]:

total_stores_data = [] # 1. 전체 장부는 루프 밖에서 딱 한 번 초기화
collected_count = 0
clean_count = 0

for sido_v in sidos_value:
    if sido_v['name'] in TARGET_SIDO_LIST:
        
        # 0. 화면 초기화 (두 번째 지역부터)
        if collected_count > 0:
            ##print("\n" + "="*50)
            region_tab = driver.find_element(By.LINK_TEXT, "지역 검색")
            region_tab.click()
            time.sleep(5)

        # 1. 지역 및 전체 버튼 클릭
        print("\n" + "="*50)
        sido_selector = f"a.set_sido_cd_btn[data-sidocd='{sido_v['code']}']"
        driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, sido_selector))
        print(f"[SEARCH] '{sido_v['name']}' 지역 데이터 수집 시작")
        
        driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, "a.set_gugun_cd_btn[data-guguncd='']"))
        print(f"[LOAD] '{sido_v['name']}' 전체 리스트 로딩 중 (10초 대기)...")
        time.sleep(10)
        
        # 2. 매장 데이터 가져오기
        stores = driver.find_elements(By.CSS_SELECTOR, "li[data-name]")
        raw_count = len(stores)
        
        # 3. 상세 정보 추출 및 중복 제거
        unique_stores = {}
        
        print(f"[LOAD] '{sido_v['name']}' 지역 전체 매장 상세 정보를 불러오는 중입니다. 잠시만 기다려주세요 ...")
        for store in stores:
            store_code = store.get_attribute("data-storecd")
            if store_code not in unique_stores:
                name = store.get_attribute("data-name")
                
                # 주소 추출
                try:
                    # 원본 데이터 가져오기 (HTML 안의 모든 텍스트 추출)
                    full_info = store.find_element(By.CSS_SELECTOR, "p.result_details").get_attribute("innerText")
                    
                    address = full_info.split('\n')[0]
                    '''
                    # 노이즈 제거
                    address = address.replace('\t', ' ').replace('\n', ' ').replace('\r', ' ')
                    '''
                    
                    # 전화번호 제거
                    if "1522-3232" in address:
                        address = address.split("1522-3232")[0]
                        clean_count += 1
                    
                    # 불필요한 공백 정리
                    address = address.strip()

                except Exception as e:
                    print(f"[ERROR] 주소 추출 중 오류 발생: {e}")
                    address = "주소 확인 불가"

                # 타입 추출
                icon_class = store.find_element(By.TAG_NAME, "i").get_attribute("class")
                s_type = "generalDT" if "pin_generalDT" in icon_class else ("reserve" if "pin_reserve" in icon_class else "general")
                
                # 딕셔너리에 저장
                unique_stores[store_code] = {
                    "매장명": name, "주소": address, "매장타입": s_type
                }

        # 통합
        final_region_list = list(unique_stores.values())
        total_stores_data.extend(final_region_list)
        
        # 로그 출력
        print(f"[VALIDATE] 중복 제거: {raw_count}개 -> {len(final_region_list)}개")
        print(f"[CLEAN] 번호 정제 매장 : {clean_count}개")
        print(f"[SUCCESS] {sido_v['name']} 지역 매장 정보 수집 완료! (누적: {len(total_stores_data)}개)")
        
        collected_count += 1


[SEARCH] '서울' 지역 데이터 수집 시작
[LOAD] '서울' 전체 리스트 로딩 중 (10초 대기)...
[LOAD] '서울' 지역 전체 매장 상세 정보를 불러오는 중입니다. 잠시만 기다려주세요 ...
[VALIDATE] 중복 제거: 705개 -> 675개
[CLEAN] 번호 정제 매장 : 30개
[SUCCESS] 서울 지역 매장 정보 수집 완료! (누적: 675개)


In [55]:
import csv
# 5. CSV 저장 (모든 반복이 끝난 후)
if total_stores_data:
    df = pd.DataFrame(total_stores_data)
    
    #df['주소'] = df['주소'].str.replace(',', '.', regex=False)
    # 이름.csv
    filename = "신은비.csv"
    
    # utf-8-sig로 저장
    df.to_csv(filename, index=False, sep=',', encoding="utf-8-sig")
    '''df.to_csv(filename, 
              index=False, 
              sep=',', 
              quoting=csv.QUOTE_NONE, 
              escapechar=' ',  # 쉼표 보호용 미세 공백
              encoding="utf-8-sig")'''
    print(f"[SAVE] {filename} (총 {len(total_stores_data)}개 매장)")
else:
    print("[WARN] 수집된 데이터가 없습니다.")

PermissionError: [Errno 13] Permission denied: '신은비.csv'

In [ ]:
driver.quit()